In [1]:
import pandas as pd
import re

In [10]:
# -------------------- Parameters --------------------
file_path = "../../solutions/bks/Results-Final-paper.xls"
sheet_name = "AvgBPPC"

# -------------------- Load Excel --------------------
df = pd.read_excel(file_path, sheet_name=sheet_name, header=[0, 1])

# -------------------- Identify columns --------------------
# First column = instance names
instance_col = df.columns[0]

# BKS columns (multi-index)
bks_value_col = ("BKS Litterature", "Value")
bks_opt_col   = ("BKS Litterature", "Is Opt ?")

# -------------------- Rename instances --------------------
def rename_instance(name):
    if pd.isna(name):
        return name

    name = str(name).strip()

    # --- Case 1: Muritiba ---
    # Format: Muritiba/BPPC_X_Y_Z
    if name.startswith("Muritiba/"):
        match = re.search(r"BPPC_(\d+)_(\d+)_(\d+)", name)
        if match:
            X = int(match.group(1))
            rest = match.group(0)  # BPPC_X_Y_Z

            if 1 <= X <= 4:
                return f"u/{rest}"
            else:
                return f"t/{rest}"

    # --- Case 2: Sandykov ---
    # Format: Sandykov/d/BPWC_...
    if name.startswith("Sandykov/"):
        parts = name.split("/", 1)
        if len(parts) == 2:
            return parts[1]  # remove "Sandykov/"

    # --- Otherwise unchanged ---
    return name

# -------------------- Build result table --------------------
result = pd.DataFrame()

# Instance names
result["Instance"] = df[instance_col].apply(rename_instance)

# -------------------- Compute fallback BKS --------------------
best_bins_cols = [col for col in df.columns if col[1] == "Best Bins"]
min_best_bins = df[best_bins_cols].min(axis=1)

# -------------------- BKS values --------------------
result["BKS"] = df[bks_value_col].copy()
result["BKS"] = result["BKS"].fillna(min_best_bins)

# -------------------- Opt column --------------------
result["Opt"] = df[bks_opt_col].fillna(2).astype(int)

# -------------------- Ensure integer BKS --------------------
result["BKS"] = result["BKS"].round().astype(int)

# -------------------- Save outputs --------------------
result.to_csv("bks_table.txt", sep=" ", index=False)
result.to_csv("bks_table.csv", index=False)

# -------------------- Preview --------------------
print("Table successfully created!\n")
print(result.head(10))

Table successfully created!

        Instance  BKS  Opt
0   u/BPPC_1_0_1   48    1
1   u/BPPC_1_0_2   49    1
2   u/BPPC_1_0_3   46    1
3   u/BPPC_1_0_4   49    1
4   u/BPPC_1_0_5   50    1
5   u/BPPC_1_0_6   48    1
6   u/BPPC_1_0_7   48    1
7   u/BPPC_1_0_8   49    1
8   u/BPPC_1_0_9   50    1
9  u/BPPC_1_0_10   46    1
